# TikTok GraphRAG Embedding Pipeline

End-to-end test harness aligned with `youtube_embedding_test.ipynb`. This notebook:

1. Reads TikTok video docs from MongoDB (`tiktok_user_video`).
2. Upserts them into Neo4j as:
   - `(:TikTokCreator { username, video_author_url })`
   - `(:TikTokVideo { video_id, video_title, video_description, video_url, create_time, view_count, like_count, comment_count, video_thumbnail_url, video_thumbnail_description, hashtag_names, voice_to_text, sticker_info_list, comment_summary_description, ... })`
   - `(:TikTokCommentTopic { video_id, name, category, platform })`
   - `(v)-[:POSTED_BY { platform: 'tiktok' }]->(c)` — **platform** distinguishes TikTok creator links from YouTube `Channel` links.
   - `(v)-[:HAS_COMMENT_TOPIC { weight, position, platform: 'tiktok' }]->(t)` — same for comment-topic edges.
3. Computes Azure OpenAI embeddings for:
   - `TikTokCommentTopic.embedding` (from `TikTokCommentTopic.name`).
   - `TikTokVideo.comment_summary_embedding` (from `comment_summary_description`).
   - `TikTokVideo.video_content_embedding` from combined text: **video_title**, **video_thumbnail_description**, **hashtag_names**, **video_description**, **voice_to_text**, **sticker_info_list**.
4. Creates Neo4j vector indexes on TikTok video fields (dedicated `tiktok_comment_topic_embedding_index` on `TikTokCommentTopic.embedding`).
5. Example semantic queries scoped to TikTok labels and indexes (`THEME` / `TOP_N` demo cells mirror the YouTube notebook).
6. Optional **similarity score analysis** (histograms + score bands) for threshold tuning — same structure as `youtube_embedding_test.ipynb`.
7. **Cleanup** cell closes the Neo4j driver and Mongo client.

Steps are **idempotent and resumable** — embeddings recompute when source text changes or is missing.

Set `MONGO_COLLECTION=tiktok_user_video` or rely on the default in the config cell below.

In [21]:
import os
import json
import time
from typing import Iterable, Iterator, List, Dict, Any, Optional

from dotenv import load_dotenv
from pymongo import MongoClient
from neo4j import GraphDatabase
from openai import AzureOpenAI, OpenAI

load_dotenv(os.getenv('DOTENV_PATH'), override=True) if os.getenv('DOTENV_PATH') else load_dotenv(override=True)
print('Env loaded from', os.getenv('DOTENV_PATH') or 'default search path')

Env loaded


In [22]:
def _redact_mongo_uri(uri: str) -> str:
    if not uri or '://' not in uri:
        return uri
    scheme, rest = uri.split('://', 1)
    if '@' not in rest:
        return uri
    creds, hostpart = rest.rsplit('@', 1)
    return f'{scheme}://***:***@{hostpart}'

MONGO_URI = os.getenv('MONGODB_URI') or os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGODB_DB') or os.getenv('MONGO_DB', 'rbl')
MONGO_COLLECTION = os.getenv('MONGO_COLLECTION', 'tiktok_user_video')


def _neo4j_uri_for_runtime(uri: str) -> str:
    """Rewrite Docker-only hostnames (`neo4j`, `host.docker.internal`) to localhost when the kernel runs on the Mac host."""
    if not uri or '://' not in uri:
        return uri
    from urllib.parse import urlparse
    p = urlparse(uri)
    host = (p.hostname or '').lower()
    if host not in {'neo4j', 'host.docker.internal'}:
        return uri
    import socket
    try:
        socket.getaddrinfo(p.hostname, p.port or 7687)
    except OSError:
        netloc = f'localhost:{p.port or 7687}'
        return p._replace(netloc=netloc).geturl()
    return uri


NEO4J_URI = _neo4j_uri_for_runtime(
    os.getenv('NEO4J_URI_DEV') or os.getenv('NEO4J_URI', 'bolt://neo4j:7687')
)
NEO4J_USER = os.getenv('NEO4J_USER_DEV') or os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD_DEV') or os.getenv('NEO4J_PASSWORD', 'neo4j')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

AZURE_OPENAI_EMBEDDING_ENDPOINT = os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_OPENAI_EMBEDDING_API_KEY = os.getenv('AZURE_OPENAI_EMBEDDING_API_KEY')
AZURE_OPENAI_EMBEDDING_API_VERSION = os.getenv('AZURE_OPENAI_EMBEDDING_API_VERSION', '2024-02-01')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = (
    os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
    or 'text-embedding-3-large'
)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-large')

MONGO_FETCH_LIMIT = int(os.getenv('MONGO_FETCH_LIMIT', '0'))
VIDEO_UPSERT_BATCH = int(os.getenv('VIDEO_UPSERT_BATCH', '200'))
EMBED_BATCH = int(os.getenv('EMBED_BATCH', '128'))
EMBED_WRITE_BATCH = int(os.getenv('EMBED_WRITE_BATCH', '500'))

TIKTOK_PLATFORM = 'tiktok'

print('Config:')
print(f'  mongo: {_redact_mongo_uri(MONGO_URI)}  db={MONGO_DB}  collection={MONGO_COLLECTION}')
print(f'  neo4j: {NEO4J_URI}  db={NEO4J_DATABASE}  user={NEO4J_USER}')
print(f'  embeddings: azure_endpoint={AZURE_OPENAI_EMBEDDING_ENDPOINT}  azure_deployment={AZURE_OPENAI_EMBEDDING_DEPLOYMENT}  azure_api_version={AZURE_OPENAI_EMBEDDING_API_VERSION}  openai_fallback={OPENAI_EMBEDDING_MODEL}')
print(f'  batches: mongo_limit={MONGO_FETCH_LIMIT}  upsert={VIDEO_UPSERT_BATCH}  embed={EMBED_BATCH}  write={EMBED_WRITE_BATCH}')

Config:
  mongo: mongodb://***:***@  db=rbl  collection=tiktok_user_video
  neo4j: bolt://neo4j:7687  db=neo4j  user=neo4j
  embeddings: azure_endpoint=https://rbtl-embedding-resource.cognitiveservices.azure.com/  azure_deployment=text-embedding-3-large  azure_api_version=2024-02-01  openai_fallback=text-embedding-3-large
  batches: mongo_limit=0  upsert=200  embed=128  write=500


In [23]:
mongo_client = MongoClient(MONGO_URI)
mongo_db = mongo_client[MONGO_DB]
mongo_coll = mongo_db[MONGO_COLLECTION]
print('Mongo collection count:', mongo_coll.estimated_document_count())

/tmp/ipykernel_156/2705439973.py:1: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mongo_client = MongoClient(MONGO_URI)


Mongo collection count: 17876


In [24]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session(database=NEO4J_DATABASE) as s:
    rec = s.run('RETURN 1 AS ok').single()
    print('Neo4j ok:', rec['ok'])
    for label in ['TikTokVideo', 'TikTokCommentTopic', 'TikTokCreator']:
        cnt = s.run(f'MATCH (n:{label}) RETURN count(n) AS c').single()['c']
        print(f'  {label}: {cnt}')

Neo4j ok: 1
  TikTokVideo: 17876
  Topic: 309047
  TikTokCreator: 110


In [25]:
def build_embedding_client():
    if AZURE_OPENAI_EMBEDDING_ENDPOINT and AZURE_OPENAI_EMBEDDING_API_KEY:
        c = AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT,
            api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
            api_version=AZURE_OPENAI_EMBEDDING_API_VERSION,
        )
        return ('azure', c, AZURE_OPENAI_EMBEDDING_DEPLOYMENT)
    if OPENAI_API_KEY:
        return ('openai', OpenAI(api_key=OPENAI_API_KEY), OPENAI_EMBEDDING_MODEL)
    raise RuntimeError(
        'No embedding credentials: set AZURE_OPENAI_EMBEDDING_ENDPOINT + '
        'AZURE_OPENAI_EMBEDDING_API_KEY (and AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME), '
        'or OPENAI_API_KEY.'
    )

EMBED_PROVIDER, embed_client, EMBED_MODEL_NAME = build_embedding_client()
print(f'Embedding provider: {EMBED_PROVIDER}  model/deployment: {EMBED_MODEL_NAME}')

probe = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=['ping'])
EMBED_DIM = len(probe.data[0].embedding)
print(f'Embedding dimensions: {EMBED_DIM}')

Embedding provider: azure  model/deployment: text-embedding-3-large
Embedding dimensions: 3072


In [26]:
def batched(iterable: Iterable, n: int) -> Iterator[list]:
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) >= n:
            yield batch
            batch = []
    if batch:
        yield batch


def embed_texts(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


def clean_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return str(value)


def _iso(v):
    if v is None:
        return None
    if hasattr(v, 'isoformat'):
        return v.isoformat()
    return str(v)


def _video_id_str(doc: dict) -> Optional[str]:
    vid = doc.get('video_id')
    if vid is None:
        return None
    if isinstance(vid, dict) and '$numberLong' in vid:
        return str(vid['$numberLong'])
    return str(vid)


def format_sticker_info_list(val: Any) -> Optional[str]:
    if val is None:
        return None
    if isinstance(val, str):
        return val.strip() or None
    if isinstance(val, list) and len(val) == 0:
        return None
    try:
        return json.dumps(val, ensure_ascii=False)
    except (TypeError, ValueError):
        return str(val)


def build_tiktok_video_content_text(
    title: Optional[str],
    thumbnail_description: Optional[str],
    hashtag_names: List[str],
    description: Optional[str],
    voice_to_text: Optional[str],
    sticker_info_list: Any,
) -> Optional[str]:
    """Combined TikTok video-side text for embedding (matches product fields)."""
    parts: List[str] = []
    if title:
        parts.append(f'Title: {title}')
    if thumbnail_description:
        parts.append(f'Thumbnail: {thumbnail_description}')
    if hashtag_names:
        parts.append(f'Hashtags: {", ".join(hashtag_names)}')
    if description:
        parts.append(f'Description: {description}')
    if voice_to_text:
        parts.append(f'Voice to text: {voice_to_text}')
    sticker_str = format_sticker_info_list(sticker_info_list)
    if sticker_str:
        parts.append(f'Stickers: {sticker_str}')
    if not parts:
        return None
    return '\n\n'.join(parts)


def normalize_mongo_doc(doc: dict) -> Dict[str, Any]:
    """Flatten one TikTok Mongo doc into a Neo4j-ready dict."""
    video_id = _video_id_str(doc)
    if not video_id:
        return {}

    topics = doc.get('comments_frequent_topics') or []
    categories = doc.get('comments_frequent_topic_categories') or []
    weights = doc.get('comments_frequent_topic_weights') or []

    topic_rows = []
    for i, name in enumerate(topics):
        name = clean_text(name)
        if not name:
            continue
        topic_rows.append({
            'name': name,
            'category': clean_text(categories[i] if i < len(categories) else None),
            'weight': float(weights[i]) if i < len(weights) and weights[i] is not None else None,
            'position': i,
        })

    video_title = clean_text(doc.get('video_title'))
    video_description = clean_text(doc.get('video_description'))
    video_thumbnail_description = clean_text(doc.get('video_thumbnail_description'))
    raw_tags = doc.get('hashtag_names') or []
    hashtag_names = [clean_text(x) for x in raw_tags if x is not None]
    hashtag_names = [h for h in hashtag_names if h]
    voice_to_text = clean_text(doc.get('voice_to_text'))
    sticker_info_list = doc.get('sticker_info_list')
    # Neo4j properties cannot store maps/lists-of-maps directly; store as serialized text.
    sticker_info_list_text = format_sticker_info_list(sticker_info_list)

    video_content_text = build_tiktok_video_content_text(
        video_title,
        video_thumbnail_description,
        hashtag_names,
        video_description,
        voice_to_text,
        sticker_info_list,
    )

    username = clean_text(doc.get('username'))
    video_url = clean_text(doc.get('video_url'))
    if not video_url and username and video_id:
        video_url = f'https://www.tiktok.com/@{username}/video/{video_id}'

    return {
        'video_id': video_id,
        'video_title': video_title,
        'video_description': video_description,
        'video_url': video_url,
        'create_time': _iso(doc.get('create_time')),
        'view_count': doc.get('view_count'),
        'like_count': doc.get('like_count'),
        'comment_count': doc.get('comment_count'),
        'video_thumbnail_url': doc.get('video_thumbnail_url'),
        'video_thumbnail_description': video_thumbnail_description,
        'video_thumbnail_keywords': [k for k in (doc.get('video_thumbnail_keywords') or []) if isinstance(k, str)],
        'hashtag_names': hashtag_names,
        'voice_to_text': voice_to_text,
        'sticker_info_list': sticker_info_list_text,
        'region_code': doc.get('region_code'),
        'video_duration': doc.get('video_duration'),
        'comment_summary_description': clean_text(doc.get('comment_summary_description')),
        'video_content_text': video_content_text,
        'username': username,
        'video_author_url': doc.get('video_author_url'),
        'topics': topic_rows,
        'platform': TIKTOK_PLATFORM,
    }

In [27]:
def ensure_schema(driver):
    stmts = [
        'CREATE CONSTRAINT tiktok_video_id_unique IF NOT EXISTS FOR (v:TikTokVideo) REQUIRE v.video_id IS UNIQUE',
        'CREATE CONSTRAINT tiktok_creator_username_unique IF NOT EXISTS FOR (c:TikTokCreator) REQUIRE c.username IS UNIQUE',
        'CREATE CONSTRAINT tiktok_comment_topic_video_name IF NOT EXISTS FOR (t:TikTokCommentTopic) REQUIRE (t.video_id, t.name) IS UNIQUE',
    ]
    with driver.session(database=NEO4J_DATABASE) as s:
        for stmt in stmts:
            s.run(stmt)
    print('Schema constraints ensured.')

ensure_schema(driver)

Schema constraints ensured.


In [28]:
UPSERT_VIDEO_CYPHER = '''
UNWIND $rows AS row

// Creator
FOREACH (_ IN CASE WHEN row.username IS NULL THEN [] ELSE [1] END |
  MERGE (c:TikTokCreator {username: row.username})
  SET c.video_author_url = coalesce(row.video_author_url, c.video_author_url)
)

MERGE (v:TikTokVideo {video_id: row.video_id})
SET v.video_title                 = row.video_title,
    v.video_description            = row.video_description,
    v.video_url                    = row.video_url,
    v.create_time                  = row.create_time,
    v.view_count                   = row.view_count,
    v.like_count                   = row.like_count,
    v.comment_count                = row.comment_count,
    v.video_thumbnail_url          = row.video_thumbnail_url,
    v.video_thumbnail_description   = row.video_thumbnail_description,
    v.video_thumbnail_keywords      = row.video_thumbnail_keywords,
    v.hashtag_names                = row.hashtag_names,
    v.voice_to_text                = row.voice_to_text,
    v.sticker_info_list            = row.sticker_info_list,
    v.region_code                  = row.region_code,
    v.video_duration               = row.video_duration,
    v.username                     = row.username,
    v.platform                     = row.platform

WITH row, v
FOREACH (_ IN CASE WHEN row.comment_summary_description IS NOT NULL AND row.comment_summary_description <> coalesce(v.comment_summary_description, '') THEN [1] ELSE [] END |
  SET v.comment_summary_description = row.comment_summary_description,
      v.comment_summary_embedding   = NULL
)
FOREACH (_ IN CASE WHEN row.comment_summary_description IS NOT NULL AND v.comment_summary_description IS NULL THEN [1] ELSE [] END |
  SET v.comment_summary_description = row.comment_summary_description
)

WITH row, v
FOREACH (_ IN CASE WHEN row.video_content_text IS NOT NULL AND row.video_content_text <> coalesce(v.video_content_text, '') THEN [1] ELSE [] END |
  SET v.video_content_text    = row.video_content_text,
      v.video_content_embedding = NULL
)
FOREACH (_ IN CASE WHEN row.video_content_text IS NOT NULL AND v.video_content_text IS NULL THEN [1] ELSE [] END |
  SET v.video_content_text = row.video_content_text
)

WITH row, v
OPTIONAL MATCH (c:TikTokCreator {username: row.username})
FOREACH (_ IN CASE WHEN c IS NULL THEN [] ELSE [1] END |
  MERGE (v)-[pb:POSTED_BY]->(c)
  SET pb.platform = $tiktok_platform
)

WITH row, v
OPTIONAL MATCH (v)-[r:HAS_COMMENT_TOPIC]->()
DELETE r

WITH row, v
UNWIND row.topics AS topic
MERGE (t:TikTokCommentTopic {video_id: row.video_id, name: topic.name})
SET t.category = topic.category,
    t.platform = $tiktok_platform
MERGE (v)-[rel:HAS_COMMENT_TOPIC]->(t)
SET rel.weight = topic.weight,
    rel.position = topic.position,
    rel.platform = $tiktok_platform
RETURN count(*) AS topic_rows
'''


def upsert_videos_to_neo4j(driver, docs_iter):
    total_videos = 0
    total_topic_rows = 0
    for batch in batched(docs_iter, VIDEO_UPSERT_BATCH):
        rows = [normalize_mongo_doc(d) for d in batch]
        rows = [r for r in rows if r.get('video_id')]
        if not rows:
            continue
        with driver.session(database=NEO4J_DATABASE) as s:
            result = s.run(
                UPSERT_VIDEO_CYPHER,
                rows=rows,
                tiktok_platform=TIKTOK_PLATFORM,
            )
            topic_rows = sum(rec['topic_rows'] for rec in result)
        total_videos += len(rows)
        total_topic_rows += topic_rows
        print(f'  upserted videos={total_videos}  topic_rows={total_topic_rows}')
    return total_videos, total_topic_rows

In [29]:
MONGO_PROJECTION = {
    'video_id': 1, 'video_title': 1, 'video_description': 1, 'video_url': 1,
    'create_time': 1, 'view_count': 1, 'like_count': 1, 'comment_count': 1,
    'video_thumbnail_url': 1, 'video_thumbnail_description': 1, 'video_thumbnail_keywords': 1,
    'hashtag_names': 1, 'voice_to_text': 1, 'sticker_info_list': 1,
    'region_code': 1, 'video_duration': 1,
    'username': 1, 'video_author_url': 1,
    'comment_summary_description': 1,
    'comments_frequent_topics': 1, 'comments_frequent_topic_categories': 1, 'comments_frequent_topic_weights': 1,
}

query = {'video_id': {'$exists': True}}
cursor = mongo_coll.find(query, MONGO_PROJECTION, no_cursor_timeout=True)
if MONGO_FETCH_LIMIT:
    cursor = cursor.limit(MONGO_FETCH_LIMIT)

start = time.time()
try:
    videos, topic_rows = upsert_videos_to_neo4j(driver, cursor)
finally:
    cursor.close()
print(f'Done. videos={videos}  topic_rows={topic_rows}  took {time.time()-start:.1f}s')

with driver.session(database=NEO4J_DATABASE) as s:
    for q, label in [
        ('MATCH (v:TikTokVideo) RETURN count(v) AS c', 'TikTokVideo'),
        ('MATCH (c:TikTokCreator) RETURN count(c) AS c', 'TikTokCreator'),
        ('MATCH (t:TikTokCommentTopic) WHERE t.platform = $p RETURN count(t) AS c', 'TikTokCommentTopic_tiktok'),
        ('MATCH ()-[r:HAS_COMMENT_TOPIC]->() WHERE r.platform = $p RETURN count(r) AS c', 'HAS_COMMENT_TOPIC_tiktok'),
        ('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL RETURN count(v) AS c', 'videos_with_summary'),
    ]:
        if 'TikTokCommentTopic' in label or 'HAS_COMMENT' in label:
            print(f'{label}: {s.run(q, p=TIKTOK_PLATFORM).single()["c"]}')
        else:
            print(f'{label}: {s.run(q).single()["c"]}')

/home/airflow/.local/lib/python3.12/site-packages/pymongo/synchronous/collection.py:1954: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


  upserted videos=200  topic_rows=8817
  upserted videos=400  topic_rows=21819
  upserted videos=600  topic_rows=33610
  upserted videos=800  topic_rows=45604
  upserted videos=1000  topic_rows=57695
  upserted videos=1200  topic_rows=72878
  upserted videos=1400  topic_rows=88996
  upserted videos=1600  topic_rows=106091
  upserted videos=1800  topic_rows=120692
  upserted videos=2000  topic_rows=127446
  upserted videos=2200  topic_rows=138854
  upserted videos=2400  topic_rows=153496
  upserted videos=2600  topic_rows=169220
  upserted videos=2800  topic_rows=188513
  upserted videos=3000  topic_rows=197262
  upserted videos=3200  topic_rows=206525
  upserted videos=3400  topic_rows=213826
  upserted videos=3600  topic_rows=219429
  upserted videos=3800  topic_rows=225701
  upserted videos=4000  topic_rows=227083
  upserted videos=4200  topic_rows=232282
  upserted videos=4400  topic_rows=242922
  upserted videos=4600  topic_rows=253792
  upserted videos=4800  topic_rows=265567
  up

In [30]:
# Vector indexes: TikTok comment topics use tiktok_comment_topic_embedding_index on :TikTokCommentTopic.embedding
TIKTOK_SUMMARY_INDEX = 'tiktok_video_summary_embedding_index'
TIKTOK_CONTENT_INDEX = 'tiktok_video_content_embedding_index'

def ensure_vector_indexes(driver, dim: int):
    stmts = [
        f'''CREATE VECTOR INDEX tiktok_comment_topic_embedding_index IF NOT EXISTS
            FOR (t:TikTokCommentTopic) ON (t.embedding)
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}''',
        f'''CREATE VECTOR INDEX {TIKTOK_SUMMARY_INDEX} IF NOT EXISTS
            FOR (v:TikTokVideo) ON (v.comment_summary_embedding)
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}''',
        f'''CREATE VECTOR INDEX {TIKTOK_CONTENT_INDEX} IF NOT EXISTS
            FOR (v:TikTokVideo) ON (v.video_content_embedding)
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}''',
    ]
    with driver.session(database=NEO4J_DATABASE) as s:
        for stmt in stmts:
            s.run(stmt)
        rows = list(s.run('SHOW VECTOR INDEXES YIELD name, labelsOrTypes, properties, state, options'))
    for r in rows:
        d = dict(r)
        if any(x in str(d.get('name', '')) for x in ('tiktok_comment_topic_embedding', 'tiktok_video')):
            print(d)

ensure_vector_indexes(driver, EMBED_DIM)

{'name': 'tiktok_video_content_embedding_index', 'labelsOrTypes': ['TikTokVideo'], 'properties': ['video_content_embedding'], 'state': 'ONLINE', 'options': {'indexConfig': {'vector.dimensions': 3072, 'vector.hnsw.m': 16, 'vector.quantization.enabled': True, 'vector.similarity_function': 'COSINE', 'vector.hnsw.ef_construction': 100}, 'indexProvider': 'vector-3.0'}}
{'name': 'tiktok_video_summary_embedding_index', 'labelsOrTypes': ['TikTokVideo'], 'properties': ['comment_summary_embedding'], 'state': 'ONLINE', 'options': {'indexConfig': {'vector.dimensions': 3072, 'vector.hnsw.m': 16, 'vector.quantization.enabled': True, 'vector.similarity_function': 'COSINE', 'vector.hnsw.ef_construction': 100}, 'indexProvider': 'vector-3.0'}}
{'name': 'topic_embedding_index', 'labelsOrTypes': ['Topic'], 'properties': ['embedding'], 'state': 'ONLINE', 'options': {'indexConfig': {'vector.dimensions': 3072, 'vector.hnsw.m': 16, 'vector.quantization.enabled': True, 'vector.similarity_function': 'COSINE', '

In [31]:
TOPIC_FETCH_CYPHER = '''
MATCH (t:TikTokCommentTopic)
WHERE t.platform = $platform
  AND t.embedding IS NULL AND t.name IS NOT NULL
RETURN id(t) AS node_id, t.name AS name
LIMIT $limit
'''

TOPIC_WRITE_CYPHER = '''
UNWIND $rows AS row
MATCH (t) WHERE id(t) = row.node_id
CALL db.create.setNodeVectorProperty(t, 'embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_topic_nodes(driver, page_size: int = 2000):
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(TOPIC_FETCH_CYPHER, limit=page_size, platform=TIKTOK_PLATFORM))
        if not pending:
            break
        texts = [r['name'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [
            {'node_id': pending[i]['node_id'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, EMBED_WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(TOPIC_WRITE_CYPHER, rows=chunk).single()['written']
            total_written += written
        print(f'  topic embeddings written so far: {total_written}  (elapsed {time.time()-start:.1f}s)')
    print(f'Done. TikTok topic embeddings written: {total_written}')

embed_topic_nodes(driver)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=5, column=8, offset=102>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 102, 'line': 5, 'column': 8}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (t:Topic)\nWHERE t.platform = $platform\n  AND t.embedding IS NULL AND t.name IS NOT NULL\nRETURN id(t) AS node_id, t.name AS name\nLIMIT $limit\n'


Done. TikTok topic embeddings written: 0


In [32]:
VIDEO_FETCH_CYPHER = '''
MATCH (v:TikTokVideo)
WHERE v.comment_summary_description IS NOT NULL
  AND v.comment_summary_embedding IS NULL
RETURN id(v) AS node_id, v.comment_summary_description AS text
LIMIT $limit
'''

VIDEO_WRITE_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE id(v) = row.node_id
CALL db.create.setNodeVectorProperty(v, 'comment_summary_embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_video_summaries(driver, page_size: int = 500):
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(VIDEO_FETCH_CYPHER, limit=page_size))
        if not pending:
            break
        texts = [r['text'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [
            {'node_id': pending[i]['node_id'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, EMBED_WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(VIDEO_WRITE_CYPHER, rows=chunk).single()['written']
            total_written += written
        print(f'  TikTok summary embeddings written so far: {total_written}  (elapsed {time.time()-start:.1f}s)')
    print(f'Done. TikTok video summary embeddings written: {total_written}')

embed_video_summaries(driver)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=5, column=8, offset=120>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 120, 'line': 5, 'column': 8}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.comment_summary_description IS NOT NULL\n  AND v.comment_summary_embedding IS NULL\nRETURN id(v) AS node_id, v.comment_summary_description AS text\nLIMIT $limit\n'


Done. TikTok video summary embeddings written: 0


In [33]:
CONTENT_FETCH_CYPHER = '''
MATCH (v:TikTokVideo)
WHERE v.video_content_text IS NOT NULL
  AND v.video_content_embedding IS NULL
RETURN id(v) AS node_id, v.video_content_text AS text
LIMIT $limit
'''

CONTENT_WRITE_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE id(v) = row.node_id
CALL db.create.setNodeVectorProperty(v, 'video_content_embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_video_content(driver, page_size: int = 500):
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(CONTENT_FETCH_CYPHER, limit=page_size))
        if not pending:
            break
        texts = [r['text'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [
            {'node_id': pending[i]['node_id'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, EMBED_WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(CONTENT_WRITE_CYPHER, rows=chunk).single()['written']
            total_written += written
        print(f'  TikTok content embeddings written so far: {total_written}  (elapsed {time.time()-start:.1f}s)')
    print(f'Done. TikTok video content embeddings written: {total_written}')

embed_video_content(driver)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=5, column=8, offset=109>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 109, 'line': 5, 'column': 8}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_text IS NOT NULL\n  AND v.video_content_embedding IS NULL\nRETURN id(v) AS node_id, v.video_content_text AS text\nLIMIT $limit\n'


Done. TikTok video content embeddings written: 0


## Test queries (TikTok-scoped)

Same retrieval primitives as `youtube_embedding_test.ipynb`, adapted for **`TikTokVideo`**, **`TikTokCreator`**, and **`tiktok_video_*`** vector indexes. Each takes a theme (e.g. `"cleaning hacks"`, `"product review"`), embeds it, then hits Neo4j.

- **`top_creators(theme)`** – vector search over `TikTokCommentTopic.embedding` (topics with `platform = 'tiktok'`), aggregate weighted contribution per **`TikTokCreator`** / username. Best for *which creators’ audiences talk about X in comments*.
- **`example_videos(theme)`** – vector search over `TikTokCommentTopic.embedding`, return **`TikTokVideo`** rows whose comment-topics match.
- **`comment_discussions(theme)`** – vector search over **`tiktok_video_summary_embedding_index`** (`comment_summary_embedding`). Best for *whose comment section discusses X*.
- **`content_videos(theme)`** – vector search over **`tiktok_video_content_embedding_index`** (combined title, thumbnail description, hashtags, description, voice-to-text, stickers).
- **`content_top_creators(theme)`** – aggregates `content_videos` hits by creator.
- **`unified_search(theme)`** – runs all three signals and fuses with Reciprocal Rank Fusion (RRF).

In [34]:
def embed_query(text: str) -> List[float]:
    return embed_texts([text])[0]


TOP_CREATORS_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE t.platform = $platform
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE rel.platform = $platform
OPTIONAL MATCH (v)-[pb:POSTED_BY {platform: $platform}]->(c:TikTokCreator)
WITH
  coalesce(c.username, v.username, 'unknown') AS creator,
  v.username AS username,
  t, v, rel, score
WITH
  creator,
  username,
  sum(score * coalesce(rel.weight, 1.0)) AS relevance,
  count(DISTINCT v) AS video_count,
  collect(DISTINCT t.name)[0..5] AS sample_topics,
  collect(DISTINCT {video_id: v.video_id, title: v.video_title, url: v.video_url})[0..8] AS sample_videos
RETURN creator, username, relevance, video_count, sample_topics, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''

EXAMPLE_VIDEOS_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE t.platform = $platform
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE rel.platform = $platform
WITH v, t, rel, score
ORDER BY score * coalesce(rel.weight, 1.0) DESC
WITH v,
     collect({topic: t.name, weight: rel.weight, score: score})[0..3] AS matches,
     max(score * coalesce(rel.weight, 1.0)) AS best
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.username, '') AS creator,
       v.video_url AS url,
       v.view_count AS views,
       best AS relevance,
       matches
ORDER BY best DESC
LIMIT $top_n
'''

COMMENT_DISCUSSIONS_CYPHER = f'''
CALL db.index.vector.queryNodes('{TIKTOK_SUMMARY_INDEX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.username, '') AS creator,
       v.video_url AS url,
       score,
       v.comment_summary_description AS comment_summary_description
ORDER BY score DESC
LIMIT $top_n
'''

CONTENT_VIDEOS_CYPHER = f'''
CALL db.index.vector.queryNodes('{TIKTOK_CONTENT_INDEX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.username, '') AS creator,
       v.video_url AS url,
       v.view_count AS views,
       v.video_thumbnail_url AS thumbnail_url,
       v.video_thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.hashtag_names AS hashtag_names,
       score
ORDER BY score DESC
LIMIT $top_n
'''

CONTENT_TOP_CREATORS_CYPHER = f'''
CALL db.index.vector.queryNodes('{TIKTOK_CONTENT_INDEX}', $k, $q) YIELD node AS v, score
OPTIONAL MATCH (v)-[pb:POSTED_BY {{platform: $platform}}]->(c:TikTokCreator)
WITH coalesce(c.username, v.username, 'unknown') AS creator,
     v.username AS username,
     v, score
WITH creator, username,
     sum(score) AS relevance,
     count(DISTINCT v) AS video_count,
     collect({{video_id: v.video_id, title: v.video_title, url: v.video_url, score: score}})[0..5] AS sample_videos
RETURN creator, username, relevance, video_count, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''


def top_creators(theme: str, top_n: int = 10, k: int = 200):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            TOP_CREATORS_CYPHER, q=q, k=k, top_n=top_n, platform=TIKTOK_PLATFORM
        )]

def example_videos(theme: str, top_n: int = 10, k: int = 100):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            EXAMPLE_VIDEOS_CYPHER, q=q, k=k, top_n=top_n, platform=TIKTOK_PLATFORM
        )]

def comment_discussions(theme: str, top_n: int = 10, k: int = 20):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(COMMENT_DISCUSSIONS_CYPHER, q=q, k=max(k, top_n), top_n=top_n)]

def content_videos(theme: str, top_n: int = 10, k: int = 50):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(CONTENT_VIDEOS_CYPHER, q=q, k=max(k, top_n), top_n=top_n)]

def content_top_creators(theme: str, top_n: int = 10, k: int = 200):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            CONTENT_TOP_CREATORS_CYPHER, q=q, k=k, top_n=top_n, platform=TIKTOK_PLATFORM
        )]


UNIFIED_CONTENT_Q = f'''
CALL db.index.vector.queryNodes('{TIKTOK_CONTENT_INDEX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, score
'''

UNIFIED_SUMMARY_Q = f'''
CALL db.index.vector.queryNodes('{TIKTOK_SUMMARY_INDEX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, score
'''

UNIFIED_TOPIC_Q = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE t.platform = $platform
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE rel.platform = $platform
WITH v.video_id AS video_id, max(score * coalesce(rel.weight, 1.0)) AS score
RETURN video_id, score
'''

UNIFIED_HYDRATE = '''
UNWIND $video_ids AS vid
MATCH (v:TikTokVideo {video_id: vid})
OPTIONAL MATCH (v)-[pb:POSTED_BY {platform: $platform}]->(c:TikTokCreator)
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(c.username, v.username) AS creator,
       v.username AS username,
       v.video_url AS url,
       v.view_count AS views,
       v.video_thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.comment_summary_description AS comment_summary_description
'''


def _rrf_rank(rows, id_key: str = 'video_id', k_const: int = 60) -> Dict[str, float]:
    scores: Dict[str, float] = {}
    for rank, row in enumerate(rows):
        scores[row[id_key]] = 1.0 / (k_const + rank + 1)
    return scores


def unified_search(theme: str, top_n: int = 10, k_each: int = 50, include_creator_rollup: bool = True):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        content_rows = [dict(r) for r in s.run(UNIFIED_CONTENT_Q, q=q, k=k_each)]
        summary_rows = [dict(r) for r in s.run(UNIFIED_SUMMARY_Q, q=q, k=k_each)]
        topic_rows = [dict(r) for r in s.run(
            UNIFIED_TOPIC_Q, q=q, k=k_each * 4, platform=TIKTOK_PLATFORM
        )]

        content_rank = _rrf_rank(content_rows)
        summary_rank = _rrf_rank(summary_rows)
        topic_rank = _rrf_rank(topic_rows)

        fused: Dict[str, float] = {}
        signals: Dict[str, List[str]] = {}
        for ranks, label in [(content_rank, 'content'), (summary_rank, 'comments'), (topic_rank, 'topic')]:
            for vid, s_val in ranks.items():
                fused[vid] = fused.get(vid, 0.0) + s_val
                signals.setdefault(vid, []).append(label)

        top_ids = sorted(fused.keys(), key=lambda x: fused[x], reverse=True)[:top_n]
        if not top_ids:
            return {'videos': [], 'creators': []}

        hydrated = {
            r['video_id']: dict(r)
            for r in s.run(UNIFIED_HYDRATE, video_ids=top_ids, platform=TIKTOK_PLATFORM)
        }
        videos = []
        for vid in top_ids:
            if vid not in hydrated:
                continue
            row = hydrated[vid]
            row['fused_score'] = round(fused[vid], 5)
            row['matched_signals'] = signals[vid]
            videos.append(row)

        creators = []
        if include_creator_rollup:
            rollup: Dict[str, Dict[str, Any]] = {}
            for row in videos:
                cid = row['username'] or row['creator'] or 'unknown'
                agg = rollup.setdefault(cid, {
                    'username': cid,
                    'creator': row['creator'],
                    'relevance': 0.0,
                    'video_count': 0,
                    'videos': [],
                })
                agg['relevance'] += row['fused_score']
                agg['video_count'] += 1
                agg['videos'].append({
                    'video_id': row['video_id'],
                    'title': row['title'],
                    'url': row.get('url'),
                })
            creators = sorted(rollup.values(), key=lambda x: x['relevance'], reverse=True)

    return {'videos': videos, 'creators': creators}

## Similarity score analysis (production thresholds)

For each **query string**, run vector search against the TikTok-specific indexes **`tiktok_video_content_embedding`**, **`tiktok_video_summary_embedding`**, and **`tiktok_comment_topic_embedding_index`** (rows restricted to **`t.platform = 'tiktok'`**). For each index:

- print **query name** + index name
- show an **ASCII histogram** of cosine similarity `score` (20 bins on \[0, 1\], with spill counts outside that range)
- print up to **5 example rows** per score band: **0.90–1.00**, **0.80–0.89**, **0.70–0.79**, **0.60–0.69**, **0.50–0.59**

Tune `SIMILARITY_K_FRACTION` (default **0.1** = 10% in prose; env default is often **0.2** unless overridden), `SIMILARITY_K_MIN` / `SIMILARITY_K_MAX`, optional fixed override `SIMILARITY_ANALYSIS_K`, and `QUERIES_FOR_THRESHOLD_TUNING`, then run the next cell.

Requires prior cells through **vector indexes** (`tiktok_video_*` index names) and **`embed_query`** from the query-helpers cell.

In [36]:
import os
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple

# Align names with ensure_vector_indexes cell
TIKTOK_CONTENT_EMBED_IDX = 'tiktok_video_content_embedding_index'
TIKTOK_SUMMARY_EMBED_IDX = 'tiktok_video_summary_embedding_index'

SIMILARITY_K_FRACTION = float(os.getenv('SIMILARITY_K_FRACTION', '0.2'))
SIMILARITY_K_MIN = int(os.getenv('SIMILARITY_K_MIN', '50'))
SIMILARITY_K_MAX = int(os.getenv('SIMILARITY_K_MAX', '25000'))

SIMILARITY_K_OVERRIDE_RAW = os.getenv('SIMILARITY_ANALYSIS_K', '').strip()

_INDEXED_NODE_COUNTS_CYPHER: Dict[str, str] = {
    TIKTOK_CONTENT_EMBED_IDX: (
        'MATCH (v:TikTokVideo) WHERE v.video_content_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    TIKTOK_SUMMARY_EMBED_IDX: (
        'MATCH (v:TikTokVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    'tiktok_comment_topic_embedding_index': (
        "MATCH (t:TikTokCommentTopic) WHERE t.embedding IS NOT NULL AND t.platform = 'tiktok' RETURN count(t) AS c"
    ),
}


def _indexed_node_counts() -> Dict[str, int]:
    out: Dict[str, int] = {}
    with driver.session(database=NEO4J_DATABASE) as s:
        for index_name, cy in _INDEXED_NODE_COUNTS_CYPHER.items():
            out[index_name] = int(s.run(cy).single()['c'])
    return out


def _k_for_index(index_name: str, counts: Dict[str, int]) -> int:
    if SIMILARITY_K_OVERRIDE_RAW:
        return max(SIMILARITY_K_MIN, int(SIMILARITY_K_OVERRIDE_RAW))
    n = counts.get(index_name, 0)
    k = int(SIMILARITY_K_FRACTION * n)
    return max(SIMILARITY_K_MIN, min(SIMILARITY_K_MAX, k))


QUERIES_FOR_THRESHOLD_TUNING = [
    'mental health',
    # 'grwm',
    # 'product review',
]

CONTENT_SCORES_CYPHER = f'''
CALL db.index.vector.queryNodes('{TIKTOK_CONTENT_EMBED_IDX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

SUMMARY_SCORES_CYPHER = f'''
CALL db.index.vector.queryNodes('{TIKTOK_SUMMARY_EMBED_IDX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

TOPIC_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE t.platform = $platform
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE rel.platform = $platform
RETURN t.name AS topic_name,
       v.video_id AS video_id, v.video_title AS title, v.video_url AS url,
       score,
       rel.weight AS topic_weight,
       rel.position AS topic_position,
       score * coalesce(rel.weight, 1.0) AS weighted_similarity
'''

SCORE_BANDS: List[Tuple[str, float, float]] = [
    ('0.90–1.00', 0.9, 1.0000001),
    ('0.80–0.89', 0.8, 0.9),
    ('0.70–0.79', 0.7, 0.8),
    ('0.60–0.69', 0.6, 0.7),
    ('0.50–0.59', 0.5, 0.6),
]


def _fetch_rows(cypher: str, q: List[float], k: int) -> List[Dict[str, Any]]:
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(cypher, q=q, k=k)]


def _fetch_topic_score_rows(q: List[float], k: int) -> List[Dict[str, Any]]:
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(TOPIC_SCORES_CYPHER, q=q, k=k, platform=TIKTOK_PLATFORM)]


def _scores(rows: List[Dict[str, Any]]) -> List[float]:
    return [float(r['score']) for r in rows if r.get('score') is not None]


def _ascii_histogram(scores: List[float], n_bins: int = 20, width: int = 50) -> None:
    if not scores:
        print('  (no scores)')
        return
    lo, hi = min(scores), max(scores)
    lo_b, hi_b = 0.0, 1.0
    counts = [0] * n_bins
    under = over = 0
    for s in scores:
        if s < lo_b:
            under += 1
        elif s > hi_b:
            over += 1
        else:
            idx = int((s - lo_b) / (hi_b - lo_b) * n_bins)
            if idx >= n_bins:
                idx = n_bins - 1
            counts[idx] += 1
    mx = max(counts) if counts else 1
    print(f'  score min={lo:.4f} max={hi:.4f}  (binning [0,1] with {n_bins} bins; below0={under} above1={over})')
    for i, c in enumerate(counts):
        left = lo_b + (hi_b - lo_b) * i / n_bins
        right = lo_b + (hi_b - lo_b) * (i + 1) / n_bins
        bar = '#' * max(1, int(width * c / mx)) if c else ''
        print(f'  [{left:.2f},{right:.2f}): {c:4d}  {bar}')


def _band_examples(rows: List[Dict[str, Any]], label: str, lo: float, hi: float, n: int = 5) -> None:
    band = [r for r in rows if r.get('score') is not None and lo <= float(r['score']) < hi]
    band.sort(key=lambda r: float(r['score']), reverse=True)
    print(f'  band {label}: {len(band)} hits (showing top {min(n, len(band))})')
    for r in band[:n]:
        pprint(r)


def print_topic_retriever_topic_name_matches(
    rows: List[Dict[str, Any]],
    topic_name_substring: Optional[str] = None,
) -> None:
    if not topic_name_substring or not str(topic_name_substring).strip():
        return
    needle = str(topic_name_substring).strip().lower()
    matched = [r for r in rows if needle in (r.get('topic_name') or '').lower()]
    if not matched:
        return
    matched.sort(
        key=lambda r: float(
            r.get('weighted_similarity')
            if r.get('weighted_similarity') is not None
            else r.get('score') or 0.0
        ),
        reverse=True,
    )
    print('-' * 88)
    print(f'  TOPIC ROWS WITH topic_name containing {topic_name_substring.strip()!r}  (n={len(matched)})')
    for r in matched:
        pprint(r)


def analyze_similarity_for_query(
    query_name: str,
    counts: Dict[str, int],
    *,
    optional_topic_name_substring: Optional[str] = None,
) -> None:
    print('=' * 88)
    print(
        f'QUERY: {query_name!r}   (k ≈ {SIMILARITY_K_FRACTION * 100:.0f}% of indexed nodes per index, '
        f'min={SIMILARITY_K_MIN} max={SIMILARITY_K_MAX}'
        + (f", override={SIMILARITY_K_OVERRIDE_RAW!r}" if SIMILARITY_K_OVERRIDE_RAW else '')
        + ')'
    )
    print('=' * 88)
    q = embed_query(query_name)

    suites: List[Tuple[str, str]] = [
        (TIKTOK_CONTENT_EMBED_IDX, CONTENT_SCORES_CYPHER),
        (TIKTOK_SUMMARY_EMBED_IDX, SUMMARY_SCORES_CYPHER),
        ('tiktok_comment_topic_embedding_index', TOPIC_SCORES_CYPHER),
    ]

    for index_name, cypher in suites:
        k = _k_for_index(index_name, counts)
        if index_name == 'tiktok_comment_topic_embedding_index':
            rows = _fetch_topic_score_rows(q, k)
        else:
            rows = _fetch_rows(cypher, q, k)
        scores = _scores(rows)
        print('-' * 88)
        n_idx = counts.get(index_name, 0)
        print(f'INDEX: {index_name}   indexed_nodes={n_idx}   k_used={k}   rows_returned={len(rows)}')
        _ascii_histogram(scores)
        for band_label, lo, hi in SCORE_BANDS:
            _band_examples(rows, band_label, lo, hi, n=5)
        if index_name == 'tiktok_comment_topic_embedding_index' and optional_topic_name_substring:
            print_topic_retriever_topic_name_matches(rows, optional_topic_name_substring)
        print()


_COUNTS_GLOBAL = _indexed_node_counts()
print('Indexed node counts (used to size k per index):', _COUNTS_GLOBAL)

for _q in QUERIES_FOR_THRESHOLD_TUNING:
    analyze_similarity_for_query(_q, counts=_COUNTS_GLOBAL)


Indexed node counts (used to size k per index): {'tiktok_video_content_embedding_index': 17866, 'tiktok_video_summary_embedding_index': 17325, 'topic_embedding_index': 110591}
QUERY: 'mental health'   (k ≈ 20% of indexed nodes per index, min=50 max=25000)
----------------------------------------------------------------------------------------
INDEX: tiktok_video_content_embedding_index   indexed_nodes=17866   k_used=3573   rows_returned=3573
  score min=0.5708 max=0.6655  (binning [0,1] with 20 bins; below0=0 above1=0)
  [0.00,0.05):    0  
  [0.05,0.10):    0  
  [0.10,0.15):    0  
  [0.15,0.20):    0  
  [0.20,0.25):    0  
  [0.25,0.30):    0  
  [0.30,0.35):    0  
  [0.35,0.40):    0  
  [0.40,0.45):    0  
  [0.45,0.50):    0  
  [0.50,0.55):    0  
  [0.55,0.60): 3305  ##################################################
  [0.60,0.65):  264  ###
  [0.65,0.70):    4  #
  [0.70,0.75):    0  
  [0.75,0.80):    0  
  [0.80,0.85):    0  
  [0.85,0.90):    0  
  [0.90,0.95):    0  
  [

In [37]:
from pprint import pprint

THEME = 'cleaning hacks'
TOP_N = 10

print(f'THEME: {THEME}')
print(f'TOP_N: {TOP_N}')

THEME: cleaning hacks
TOP_N: 10


In [38]:
print(f'--- CONTENT: TikTok videos that depict / discuss "{THEME}" (content embedding) ---')
for row in content_videos(THEME, top_n=TOP_N):
    pprint(row)

--- CONTENT: TikTok videos that depict / discuss "cleaning hacks" (content embedding) ---
{'creator': 'myronkoops',
 'hashtag_names': ['clean', 'housecleaning', 'cleantok', 'sundayreset'],
 'score': 0.721401572227478,
 'thumbnail_description': 'An adult, presenting as female, stands in a modern '
                          'kitchen beside an open dishwasher. She is wearing a '
                          'matching blue athletic outfit, including a '
                          "sweatshirt with 'KP ACTIVE' printed on it and "
                          'shorts. She holds a colorful pouch of dishwasher '
                          'tablets with one hand while appearing to remove a '
                          'tablet with the other. Her posture and gaze are '
                          'directed towards the pouch. The environment is '
                          'bright, clean, and minimalist, featuring white '
                          'cabinetry, a marble countertop, and modern '
                

In [39]:
print(f'--- CONTENT: creators whose TikTok videos match "{THEME}" (content embedding rollup) ---')
for row in content_top_creators(THEME, top_n=TOP_N):
    pprint(row)

--- CONTENT: creators whose TikTok videos match "cleaning hacks" (content embedding rollup) ---
{'creator': 'quintymirjam',
 'relevance': 29.3105331659317,
 'sample_videos': [{'score': 0.7212727665901184,
                    'title': 'Sundays are for cleaning 🫧🤍 | @Cleanipedia NL '
                             '#CleanTok #sundayreset #Robjin #Sun ',
                    'url': 'https://www.tiktok.com/@quintymirjam/video/7290181592809524512',
                    'video_id': '7290181592809524512'},
                   {'score': 0.6944350004196167,
                    'title': 'Love this hack! #hairrollers #dyson ',
                    'url': 'https://www.tiktok.com/@quintymirjam/video/7194828226865270021',
                    'video_id': '7194828226865270021'},
                   {'score': 0.6906852126121521,
                    'title': 'Clean your brushes 🫶🏻🤢 #hairtok ',
                    'url': 'https://www.tiktok.com/@quintymirjam/video/7184063530918841606',
                    'vide

In [40]:
print(f'--- TOPICS: creators whose comment-topic graph matches "{THEME}" ---')
for row in top_creators(THEME, top_n=TOP_N):
    pprint(row)

--- TOPICS: creators whose comment-topic graph matches "cleaning hacks" ---
{'creator': 'hemanederland',
 'relevance': 1.7625695151090623,
 'sample_topics': ['Cleaning Hacks',
                   'Cleaning Tips',
                   'Bathroom Cleaning Tips',
                   'Life Hack',
                   'Alternative Cleaning Products'],
 'sample_videos': [{'title': 'Maak je kraan reflecterend met behulp van '
                             'bakpapier. Simpel en shiny! #badkamer '
                             '#schoonmaken #kraan #lifehack ',
                    'url': 'https://www.tiktok.com/@hemanederland/video/7270556613633756448',
                    'video_id': '7270556613633756448'},
                   {'title': 'Bye bye kalkaanslag! Leer hoe je je kranen kunt '
                             'laten stralen met een citroen. 🍋✨ #lifehack '
                             '#kalkaanslag #citroen',
                    'url': 'https://www.tiktok.com/@hemanederland/video/7247417170223762714

In [41]:
print(f'--- COMMENT SUMMARIES matching "{THEME}" ---')
for row in comment_discussions(THEME, top_n=TOP_N):
    pprint(row)

--- COMMENT SUMMARIES matching "cleaning hacks" ---
{'comment_summary_description': 'The comments discuss a cleaning hack, '
                                'particularly its application for glass shower '
                                'cabins, and whether it is truly effective or '
                                'simply leaves a greasy layer. Several users '
                                'mention their own preferred cleaning '
                                'products, such as fabric softener and '
                                'limescale remover, highlighting alternative '
                                'approaches. While some react positively with '
                                'emojis and supportive remarks, the overall '
                                'sentiment is mixed, with skepticism about the '
                                "hack's ability to actually clean versus just "
                                'provide superficial shine.',
 'creator': 'hemanederland',
 

In [42]:
print(f'--- UNIFIED fused search for "{THEME}" ---')
fused = unified_search(THEME, top_n=TOP_N)
print('Top videos (with which signals matched):')
for row in fused['videos']:
    pprint(row)

--- UNIFIED fused search for "cleaning hacks" ---
Top videos (with which signals matched):
{'comment_summary_description': 'Commenters are actively discussing cleaning '
                                'methods, including curiosity about specific '
                                'ingredients like lemon juice and whether '
                                'these tips work on various surfaces like '
                                'glass shower walls. Some viewers express '
                                'gratitude and eagerness to try out the '
                                'cleaning advice, while others mention plans '
                                'to buy suggested products or inquire about '
                                'alternatives. The overall sentiment is '
                                'positive, enthusiastic, and supportive, with '
                                'the main themes being practical cleaning '
                                'solutions, product questions,

In [43]:
print(f'--- UNIFIED creator rollup for "{THEME}" ---')
fused = unified_search(THEME, top_n=TOP_N)
for row in fused['creators']:
    pprint(row)

--- UNIFIED creator rollup for "cleaning hacks" ---
{'creator': 'hemanederland',
 'relevance': 0.21826,
 'username': 'hemanederland',
 'video_count': 5,
 'videos': [{'title': 'Bye bye kalkaanslag! Leer hoe je je kranen kunt laten '
                      'stralen met een citroen. 🍋✨ #lifehack #kalkaanslag '
                      '#citroen',
             'url': 'https://www.tiktok.com/@hemanederland/video/7247417170223762714',
             'video_id': '7247417170223762714'},
            {'title': 'Maak je kraan reflecterend met behulp van bakpapier. '
                      'Simpel en shiny! #badkamer #schoonmaken #kraan '
                      '#lifehack ',
             'url': 'https://www.tiktok.com/@hemanederland/video/7270556613633756448',
             'video_id': '7270556613633756448'},
            {'title': 'Tijd voor schone voegen! Leer onze geheime truc om je '
                      'badtegelvoegen weer helemaal schoon te krijgen! ✨🧽  '
                      '#badkamertegels #voeg

In [ ]:
try:
    driver.close()
except Exception:
    pass
try:
    mongo_client.close()
except Exception:
    pass
print('Closed clients.')